In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rasterstats import zonal_stats
import rasterio

FARMS_PATH   = "GeoFiles/CSB/CSB_AZ_cleaned.shp"                   
RASTER_PATH  = "GeoFiles/Soil/AZ_Soil_Taxonomy_Preprocessed.tif"   
OUT_PATH     = "GeoFiles/Soil/FarmWise/Farm_SoilTaxonomy_clean.shp"

gdf = gpd.read_file(FARMS_PATH)
print(f"Loaded {len(gdf)} farms")

#read raster to get nodata value
with rasterio.open(RASTER_PATH) as src:
    nodata = src.nodata
    raster_crs = src.crs
    print(f"Raster CRS: {raster_crs}, NoData: {nodata}")

# Compute zonal stats for each farm 
print("Running zonal_stats...")
stats = zonal_stats(
    gdf,
    RASTER_PATH,
    categorical=True,   
    nodata=nodata,  
    all_touched=True    
)

# Convert to DataFrame
df_stats = pd.DataFrame(stats)
df_stats.index = gdf.index

soil_classes = []
valid_counts = []
total_counts = []

# Determine dominant soil class for each farm
for idx, row in df_stats.iterrows():
    if row.dropna().empty:
        soil_classes.append(np.nan)
        valid_counts.append(0)
        total_counts.append(0)
        continue

    # drop NaN entries
    counts = row.dropna().astype(int)

    if counts.sum() == 0:
        soil_classes.append(np.nan)
        valid_counts.append(0)
        total_counts.append(0)
    else:
        dom = counts.idxmax()  
        soil_classes.append(dom)
        valid_counts.append(counts.sum())
        total_counts.append(counts.sum())  

# Add results to GeoDataFrame
gdf["SoilClass"]   = soil_classes
gdf["soil_valid"]  = valid_counts
gdf["soil_total"]  = total_counts
gdf["soil_covpct"] = np.where(gdf["soil_total"] > 0,
                              100 * np.array(valid_counts) / np.array(total_counts),
                              0)

# Summary of results before handling NaNs
with rasterio.open(RASTER_PATH) as src:
    arr = src.read(1)
    nodata = src.nodata
    transform = src.transform

    rows, cols = np.where(arr != nodata)
    valid_coords = np.array(rasterio.transform.xy(transform, rows, cols)).T
    valid_values = arr[rows, cols]

nan_idx = gdf[gdf["SoilClass"].isna()].index
for idx in nan_idx:
    centroid = gdf.loc[idx].geometry.centroid
    cx, cy = centroid.x, centroid.y
    d2 = (valid_coords[:,0] - cx)**2 + (valid_coords[:,1] - cy)**2
    nearest_idx = np.argmin(d2)
    gdf.at[idx, "SoilClass"] = valid_values[nearest_idx]


nan_count = gdf["SoilClass"].isna().sum()
print(f"\nNaN farms: {nan_count} ({100*nan_count/len(gdf):.2f}%)")

print("\nSoil class distribution (non-NaN):")
print(gdf["SoilClass"].value_counts().sort_index())

gdf_nan = gdf[gdf["SoilClass"].isna()]
print(gdf_nan)

gdf = gdf[['CSBID', 'CSBACRES', 'CNTY', 'geometry', 'SoilClass', 'soil_valid', 'soil_total', 'soil_covpct']].copy()

gdf.to_file(OUT_PATH, driver="ESRI Shapefile")
print(f"Saved clean farm-level soil taxonomy to {OUT_PATH}")

counts = gdf["SoilClass"].dropna().astype(int).value_counts().sort_index()
plt.figure(figsize=(10,5))
counts.plot(kind="bar", edgecolor="black")
plt.title("Distribution of Dominant Soil Classes Across Farms")
plt.xlabel("Soil Class")
plt.ylabel("Number of Farms")
plt.tight_layout()
plt.show()
